In [0]:
# This Cell is sued to get Secrets we have in Key Vaultss

client_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-client-id-app-reg")
client_secret = dbutils.secrets.get(scope="kv-scope", key="db-secret-value-appregi")
tenant_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-tenant")

# Storage account name
storage_account = "stdehealthcareanalytics"

# OAuth configs
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
# SQL Server connection

sql_server = "healthcare-project-server-2026.database.windows.net"
sql_user = "username"
sql_pass = dbutils.secrets.get(scope="kv-scope", key="sql-pwd-azureportal")
sql_db = "healthcarebootcamp"

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_db}"

connection_properties = {
    "user": sql_user,
    "password": sql_pass,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import col

# Read FilesTable
files_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.FilesTable",
    properties=connection_properties
)

# Filter patients files
pending_files_df = files_df.filter(
    (col("Status") == "Bronze_Processed") &
    (col("FileName").like("patients%"))
)

# Create batch list
batch_list = [row.FileName for row in pending_files_df.select("FileName").collect()]

print("Files to process:", batch_list)

# Read from Bronze folder
bronze_path = "abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/patients"

df = spark.read.format("parquet").load(bronze_path)

display(df.limit(50))

Files to process: []


patient_id,first_name,last_name,age,gender,city,province,postal_code,contact_number,insurance_provider,blood_type
PAT-17-0000001,Ronald,Griffin,7,M,Mississauga,ON,X4J4A4,+1 (417) 986-6843,Manulife,B-
PAT-17-0000002,Brett,Hill,42,null,Burnaby,BC,R6L5S2,566-647-2220,OHIP,AB-
PAT-17-0000003,Kathy,Wheeler,4,U,Red Deer,AB,L1C 9E6,876 605 6182,OHIP,A+
PAT-17-0000004,NULL,Choi,91,F,Vancouver,BC,C5T7H1,NULL,Â OHIP,B+
PAT-17-0000005,Susan,Mcguire,32,Other,Richmond,BC,null,712-731-4884 x525,Blue Cross,B+
null,Nathaniel,Harrison,23,Other,null,BC,P6N7R7,1 (691) 589-2731,Sun Life,AB+
PAT-17-0000007,Deborah,Murphy,51,Other,Brampton,NULL,S3P 6N1,(770) 391-0793 x254,Manulife,B-
PAT-17-0000008,Richard,Garrett,13,Other,Calgary,AB,N4M6E5,1-248-488-7477,Manulife,AB-
PAT-17-0000009,NULL,null,20,F,Richmond,BC,J4R 2E4,+1 (821) 831-6939,Sun Life,A+
null,Michael,Rivera,94,M,Edmonton,AB,null,1 (945) 131-1245,Private Pay,B+


In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

# Trim all string columns
for column in df.columns:
    df = df.withColumn(column, trim(col(column)))

# Empty → NULL
for column in df.columns:
    df = df.withColumn(column, when(col(column) == "", None).otherwise(col(column)))

# Remove weird characters
for column in df.columns:
    df = df.withColumn(
        column,
        when(col(column).isNotNull(),
             regexp_replace(col(column).cast("string"), "Â|\t", "")
        ).otherwise(col(column))
    )

# "NULL" → NULL
df = df.replace("NULL", None)

display(df.limit(10))

patient_id,first_name,last_name,age,gender,city,province,postal_code,contact_number,insurance_provider,blood_type
PAT-17-0000001,Ronald,Griffin,7,M,Mississauga,ON,X4J4A4,+1 (417) 986-6843,Manulife,B-
PAT-17-0000002,Brett,Hill,42,null,Burnaby,BC,R6L5S2,566-647-2220,OHIP,AB-
PAT-17-0000003,Kathy,Wheeler,4,U,Red Deer,AB,L1C 9E6,876 605 6182,OHIP,A+
PAT-17-0000004,null,Choi,91,F,Vancouver,BC,C5T7H1,null,OHIP,B+
PAT-17-0000005,Susan,Mcguire,32,Other,Richmond,BC,null,712-731-4884 x525,Blue Cross,B+
null,Nathaniel,Harrison,23,Other,null,BC,P6N7R7,1 (691) 589-2731,Sun Life,AB+
PAT-17-0000007,Deborah,Murphy,51,Other,Brampton,null,S3P 6N1,(770) 391-0793 x254,Manulife,B-
PAT-17-0000008,Richard,Garrett,13,Other,Calgary,AB,N4M6E5,1-248-488-7477,Manulife,AB-
PAT-17-0000009,null,null,20,F,Richmond,BC,J4R 2E4,+1 (821) 831-6939,Sun Life,A+
null,Michael,Rivera,94,M,Edmonton,AB,null,1 (945) 131-1245,Private Pay,B+


In [0]:
from pyspark.sql.functions import col

print("Writing to Silver...")

# Remove NULL primary key
df_valid = df.filter(col("patient_id").isNotNull())

# Deduplicate
df_valid = df_valid.dropDuplicates(["patient_id"])

# Silver path
silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/patients/silver"

# Write
df_valid.write.format("delta").mode("append").save(f"{silver_path}/patients_silver")

print("Write complete")

Writing to Silver...
Write complete


In [0]:
from pyspark.sql.functions import col

# Identify bad records
bad_df = df.filter(
    col("patient_id").isNull() |
    col("first_name").isNull() |
    col("last_name").isNull()
)

# Path
bad_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/patients/badrecords"

# Write bad records
bad_df.write.format("delta").mode("append").save(bad_path)

display(bad_df)

patient_id,first_name,last_name,age,gender,city,province,postal_code,contact_number,insurance_provider,blood_type
PAT-17-0000004,null,Choi,91,F,Vancouver,BC,C5T7H1,null,OHIP,B+
null,Nathaniel,Harrison,23,Other,null,BC,P6N7R7,1 (691) 589-2731,Sun Life,AB+
PAT-17-0000009,null,null,20,F,Richmond,BC,J4R 2E4,+1 (821) 831-6939,Sun Life,A+
null,Michael,Rivera,94,M,Edmonton,AB,null,1 (945) 131-1245,Private Pay,B+
PAT-17-0000015,null,Jackson,37,F,Edmonton,AB,V9S8Y2,254 938 4329,Private Pay,AB-
null,Heidi,Zavala,null,null,Lethbridge,AB,C2B8L5,1-281-886-0249,Sun Life,A+
PAT-17-0000031,null,null,48,M,Red Deer,AB,null,+1 (535) 709-6632,Blue Cross,B+
PAT-17-0000035,null,Thomas,29,F,Vancouver,BC,N4S4M6,1 (601) 310-3739,Sun Life,B+
null,Isabella,Rodriguez,2,U,Red Deer,AB,Y3L 5H9,783-651-9039,Private Pay,AB-
PAT-17-0000044,null,Tapia,37,U,Quebec City,QC,X3A4J7,null,Blue Cross,O+


In [0]:
silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/patients/silver"
spark.read.format("delta").load(f"{silver_path}/patients_silver").display()

patient_id,first_name,last_name,age,gender,city,province,postal_code,contact_number,insurance_provider,blood_type
PAT-17-0000003,Kathy,Wheeler,4,U,Red Deer,AB,L1C 9E6,876 605 6182,OHIP,A+
PAT-17-0000464,Michaela,Jackson,7,M,Brampton,ON,X7M 1V9,749-276-7389,Private Pay,O+
PAT-17-0000653,Gregory,Houston,78,U,Brampton,ON,null,+1 (799) 910-2949,Sun Life,O-
PAT-17-0000742,null,Lang,null,Other,Surrey,BC,null,819-106-2990 x509,OHIP,A-
PAT-17-0000899,Kevin,Lynch,80,U,Brampton,ON,Y8Y 9S8,(720) 703-5831,null,AB+
PAT-17-0000992,Kelly,Graham,76,M,Mississauga,null,G7Y5C9,null,Private Pay,A+
PAT-17-0001246,Matthew,Reed,57,null,Ottawa,ON,R2J 5B7,1-418-780-8191,Blue Cross,B+
PAT-17-0001275,Olivia,Ross,39,null,Surrey,BC,null,null,Private Pay,O+
PAT-17-0001528,Courtney,Ball,84,Other,Montreal,QC,J3X3X3,(710) 718-8186,OHIP,O+
PAT-17-0002770,Darrell,Richards,30,F,null,AB,Y8M 1H2,362 865 1096,Blue Cross,A+


In [0]:
# Check Bad Records
# Check Audit 
spark.read.jdbc( url=jdbc_url, table="dbo.AuditLog", properties=connection_properties ).filter("TableName = 'patients'").display()

Id,BatchName,TableName,Status,Layer,StartTime,EndTime
1,patients_batch2,Patients,Raw_Completed,Raw,2026-04-15T19:19:01.69Z,2026-04-15T19:19:39.247Z
2,patients_batch1,Patients,Raw_Completed,Raw,2026-04-15T19:18:52.253Z,2026-04-15T19:19:20.62Z
3,patients_batch3,Patients,Raw_Completed,Raw,2026-04-15T19:19:00.807Z,2026-04-15T19:19:25.62Z
4,patients_batch4,Patients,Raw_Completed,Raw,2026-04-15T19:19:02.817Z,2026-04-15T19:19:23.753Z
